In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Create a tiny dataset (Text, Sentiment Label)
data = [
    ("i love this movie", 1),
    ("this film is great", 1),
    ("i like the acting", 1),
    ("i hate this movie", 0),
    ("this film is bad", 0),
    ("i dislike the acting", 0)
]

# 2. Build a character-level vocabulary
all_chars = set("".join([text for text, _ in data]))
vocab = {char: idx + 1 for idx, char in enumerate(sorted(all_chars))} # 0 reserved for padding
vocab['<pad>'] = 0
vocab_size = len(vocab)
max_length = 20  # Pad/truncate all sequences to this size

# 3. Text to Tensor helper function
def text_to_tensor(text, vocab, max_len):
    tokens = [vocab[char] for char in text if char in vocab]
    tokens = tokens[:max_len] + [0] * (max_len - len(tokens))  # Padding
    return torch.tensor(tokens, dtype=torch.long)

# 4. PyTorch Dataset class
class TextDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.samples = [(text_to_tensor(text, vocab, max_len), torch.tensor(label, dtype=torch.float32))
                        for text, label in data]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# Create DataLoader
dataset = TextDataset(data, vocab, max_length)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MiniTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # Single standard transformer encoder block
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)      # Shape: [Batch, Seq_Len, d_model]
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)          # Global Average Pooling across sequence length
        x = self.fc(x).squeeze(-1) # Output raw logits
        return x

# Instantiate model
model = MiniTransformerClassifier(
    vocab_size=vocab_size,
    d_model=16,     # Small embedding size
    nhead=2,        # 2 Attention heads
    num_layers=1,   # 1 Transformer Layer
    num_classes=1   # Binary classification (positive/negative)
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

print("Starting training...")
model.train()
for epoch in range(50):  # 50 Epochs is enough for this micro-dataset
    epoch_loss = 0
    for texts, labels in dataloader:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {epoch_loss/len(dataloader):.4f}")

model.eval()
test_sentences = ["i love acting", "this film is bad"]

print("\n--- Predictions ---")
with torch.no_grad():
    for sentence in test_sentences:
        input_tensor = text_to_tensor(sentence, vocab, max_length).unsqueeze(0).to(device)
        logit = model(input_tensor)
        probability = torch.sigmoid(logit).item()
        sentiment = "Positive" if probability > 0.5 else "Negative"
        print(f"Text: '{sentence}' -> Sentiment: {sentiment} ({probability*100:.1f}% confidence)")


Using device: cuda
Starting training...
Epoch [10/50], Loss: 0.6591
Epoch [20/50], Loss: 0.4439
Epoch [30/50], Loss: 0.0289
Epoch [40/50], Loss: 0.0087
Epoch [50/50], Loss: 0.0044

--- Predictions ---
Text: 'i love acting' -> Sentiment: Positive (87.2% confidence)
Text: 'this film is bad' -> Sentiment: Negative (0.2% confidence)
